In [ ]:
import typing as tp
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt


def show_img(img, title=None):
    # Convert BGR to RGB for correct display
    if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    plt.figure()
    plt.imshow(img)

    # Add title if provided
    if title:
        plt.title(title)

    plt.axis('off')
    plt.tight_layout
    plt.show()
    plt.close()

def canny_blur(img, low, high):
    """
    Detect edges using cv.Canny edge detection

    Args:
        image in BGR
        low: Lower Canny threshold
        high: Upper Canny threshold

    Returns:
        np.ndarray: Binary edge image
    """
    # Convert to grayscale
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    # Reduce noise
    blur = cv.GaussianBlur(gray, (5, 5), 0)

    # Detect edges
    edges = cv.Canny(blur, low, high)

    return edges


def draw_lines(img, lines, thickness):
    """
    Draw line segments on a blank image

    Args:
        img (np.ndarray): Reference image for size
        lines: Line coordinates [x1, y1, x2, y2]

    Returns:
        np.ndarray: Image containing just drawn lines
    """
    t = thickness
    # Blank image for drawing
    out = np.zeros_like(img)

    # Return empty image if no lines detected
    if lines is None:
        print("No lines detected")
        return out

    # Ensure correct shape
    lines = np.asarray(lines).reshape(-1, 4)

    # Draw each line in green
    for x1, y1, x2, y2 in lines:
        cv.line(
            out,
            (int(x1), int(y1)),
            (int(x2), int(y2)),
            (0, 255, 0),
            t
        )

    return out


def avg_lines(img, lines):
    """
    Average detected line segments into left and right lane lines

    Args:
        img (np.ndarray)
        lines (array-like): Line segments from Hough transform

    Returns:
        np.ndarray: Averaged left and right lane lines
    """
    left = []
    right = []

    # Separate lines by slope sign
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        m, b = np.polyfit((x1, x2), (y1, y2), 1)

        if m < 0:
            left.append((m, b))
        else:
            right.append((m, b))

    # Average slope and intercept
    left_avg = np.average(left, axis=0)
    right_avg = np.average(right, axis=0)

    # Convert to coordinates
    left_line = line_coords(img, left_avg)
    right_line = line_coords(img, right_avg)

    return np.array([left_line, right_line])


def line_coords(img, params):
    """
    Convert line parameters to pixel coordinates

    Args:
        img (np.ndarray): Image used to determine bounds
        params (tuple): Line parameters (slope, intercept)

    Returns:
        np.ndarray: Line coordinates [x1, y1, x2, y2]
    """
    m, b = params

    # Vertical bounds
    y1 = img.shape[0]
    y2 = int(y1 * 3 / 5)

    # Compute x, derived by inverting y = mx + b
    x1 = int((y1 - b) / m)
    x2 = int((y2 - b) / m)

    return np.array([x1, y1, x2, y2])


In [ ]:
# ============================================================
# Road1
# ============================================================

def roi(img, triangle):
    """
    Apply a region of interest to cut the image

    Args:
        img (np.ndarray): Input image

    Returns:
        np.ndarray: Image masked to the region of interest
    """

    # Polygon defining the region of interest
    poly = triangle

    # Create mask and fill ROI
    mask = np.zeros_like(img)
    mask = cv.fillPoly(mask, poly, 255)

    # Apply mask to the image
    mask = cv.bitwise_and(img, mask)

    return mask

# Read the input image from disk
img = cv.imread('roads/road1.png')
show_img(img, title='Original image')

# Create a copy of the image to draw lane lines on
lane_img = np.copy(img)

# --------------------------------------------------------
# Hough Line Detection
# --------------------------------------------------------

#ROI
h = lane_img.shape[0]
poly = np.array([
        [(50, h), (1055, h), (845, 325)]
        ])

# Apply grayscale conversion, Gaussian blur, and Canny edge detection
edges_blur = canny_blur(img, 100, 300)

# Restrict edge detection to the region of interest
edges_roi = roi(edges_blur, poly)

# show_img(edges_roi) #debug 

# Parameters controlling the Hough Line Transform
rho = 10                    # Distance resolution in pixels
theta = np.pi / 180         # Angle resolution in radians (=1 degree)
thresh = 400                # Minimum votes

# Detect line segments using probabilistic Hough transform for computational efficiency
lines = cv.HoughLinesP(
    edges_roi,
    rho=rho,
    theta=theta,
    threshold=thresh,
    minLineLength=50,
    maxLineGap=20
)

# Draw detected line segments on a blank image
line_img = draw_lines(lane_img, lines, 5)

# Draw detected lines onto the original image by doing a weighted sum
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
show_img(all_lines_img, title='Lane lines')

# --------------------------------------------------------
# Lane Line Averaging
# --------------------------------------------------------

# Compute averaged left and right lane lines
avg = avg_lines(lane_img, lines)

# Draw the averaged lane lines on a blank image
line_img = draw_lines(img, avg, 15)

# Sum the averaged lane lines on the original image
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL')


In [ ]:
# --------------------------------------------------------
# Now we want to look only at the first lane and not the entire roadway
# --------------------------------------------------------

# Apply grayscale conversion, Gaussian blur, and Canny edge detection
edges_blur = canny_blur(img, 100, 300)

# Restrict edge detection to the region of interest
# Image height
h = lane_img.shape[0]

# Polygon defining the region of interest
poly = np.array([
        [(500, h), (1055, h), (840, 325)]
        ])
edges_roi = roi(edges_blur, poly)

# show_img(edges_roi) #debug

# Parameters controlling the Hough Line Transform
rho = 10                    # Distance resolution in pixels
theta = np.pi / 180         # Angle resolution in radians (=1 degree)
thresh = 200                # Minimum votes

# Detect line segments using probabilistic Hough transform for computational efficiency
lines = cv.HoughLinesP(
    edges_roi,
    rho=rho,
    theta=theta,
    threshold=thresh,
    minLineLength=20,
    maxLineGap=10
)

# Draw detected line segments on a blank image
line_img = draw_lines(lane_img, lines, 5)

# Draw detected lines onto the original image by doing a weighted sum
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
show_img(all_lines_img, title='Lane lines')

# --------------------------------------------------------
# Lane Line Averaging
# --------------------------------------------------------

# Compute averaged left and right lane lines
avg = avg_lines(lane_img, lines)

# Draw the averaged lane lines on a blank image
line_img = draw_lines(img, avg, 15)

# Sum the averaged lane lines on the original image
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL')


In [ ]:
# --------------------------------------------------------
# Type of lane detection (solid/dashed)
# --------------------------------------------------------

def split_lines_by_side(lines):
    """
    Split Hough-detected line segments into left and right lane,
    based on the sign of their slope

    Args:
        lines: Output of cv.HoughLinesP, each line in format
                [x1, y1, x2, y2]

    Returns:
        tuple:
            left (list): Line segments with negative slope (left lane)
            right (list): Line segments with positive slope (right lane)
    """
    left, right = [], []

    # Iterate over all detected Hough line segments
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)

        # Ignore perfectly vertical lines to avoid division by zero
        if x1 == x2:
            continue

        # Compute slope of the line segment
        slope = (y2 - y1) / (x2 - x1)

        # Negative slope → left lane, positive slope → right lane
        if slope < 0:
            left.append((x1, y1, x2, y2))
        else:
            right.append((x1, y1, x2, y2))

    return left, right


def checkLine_yc(img, n=50, y_min_ratio=0.6):
    """
    Generate evenly spaced horizontal lines y-coordinates
    over the lower part of the image (road area)

    Args:
        img (np.ndarray): Input image used to determine height
        n (int): Number of lines
        y_min_ratio (float): Fraction of image height where lines start

    Returns:
        np.ndarray: Array of y-coordinates
    """
    h = img.shape[0]
    
    # start from y_min_ratio * h, ending at the bottom h-1
    return np.linspace(
        int(h * y_min_ratio),
        h - 1,
        n
    ).astype(int)


def segment_intersects_y(line, y, tol=2):
    """
    Check whether a line segment intersects (or is very close to)
    a horizontal checkline at coordinate y

    Args:
        line (tuple): Line segment defined as (x1, y1, x2, y2)
        y (int): y-coordinate of the check line
        tol (int): Vertical tolerance in pixels

    Returns:
        bool: True if the segment intersects the check line, False otherwise
    """
    x1, y1, x2, y2 = line

    # Check if y is between the segment's vertical endpoints
    return (min(y1, y2) - tol) <= y <= (max(y1, y2) + tol)


def checkLine_intersects(lines, yc):
    """
    For each check line, determine whether at least one line segment
    intersects it

    Args:
        lines (list): List of Hough line segments for one lane side
        yc (np.ndarray): y-coordinates of check lines

    Returns:
        np.ndarray: Boolean array where True means a check line
                    intersects at least one segment
    """
    hits = []

    # Evaluate each check line independently
    for y in yc:
        hit = False

        # Check all line segments for intersection with this check line
        for line in lines:
            if segment_intersects_y(line, y):
                hit = True
                break

        hits.append(hit)

    return np.array(hits)


def classify_line_type(lines, img, n_checkLines=50):
    """
    Classify a lane marking as solid or dashed based on check line
    intersections with raw Hough line segments

    A solid lane intersects most check lines, while a dashed lane
    has repeated gaps

    Args:
        lines (list): Hough line segments belonging to one lane side
        img (np.ndarray): Original image (used for size reference)
        n_check lines (int): Number of horizontal check lines

    Returns:
        tuple:
            (string): "solid" or "dashed"
            hit_ratio (float): Fraction of check lines intersected
            gap_count (int): Number of check lines with no intersection
    """
    # Generate check line positions
    yc = checkLine_yc(img, n=n_checkLines)

    # Check which check lines intersect any Hough segment
    hits = checkLine_intersects(lines, yc)

    # Ratio of check lines that intersect at least one segment
    hit_ratio = hits.mean()

    # Number of check lines without any intersection (gaps)
    gap_count = np.sum(~hits)

    # Classification based on hit ratio
    if hit_ratio > 0.7:
        return "solid", hit_ratio, gap_count
    else:
        return "dashed", hit_ratio, gap_count


# ============================================================
# Lane type classification using Hough line segments
# ============================================================

# Separate detected Hough lines into left and right lanes
left_lines, right_lines = split_lines_by_side(lines)

# Classify left lane
left_type, left_ratio, left_gaps = classify_line_type(
    left_lines, img
)

# Classify right lane
right_type, right_ratio, right_gaps = classify_line_type(
    right_lines, img
)

#print(f"Left lane:  {left_type} | hit_ratio={left_ratio:.2f}, gaps={left_gaps}")
#print(f"Right lane: {right_type} | hit_ratio={right_ratio:.2f}, gaps={right_gaps}")

# --------------------------------------------------------
# Overlay Text Results on Image
# --------------------------------------------------------

# Define text properties
font = cv.FONT_HERSHEY_SIMPLEX
font_scale = 0.8
color = (255, 255, 255) 
thickness = 2
line_type = cv.LINE_AA

# Text for left lane
left_text = f"Left: {left_type} | Ratio: {left_ratio:.2f} | Gaps: {left_gaps}"
# Text for right lane
right_text = f"Right: {right_type} | Ratio: {right_ratio:.2f} | Gaps: {right_gaps}"

# Draw the text on the 'final' image
# (50, 50) is the (x, y) coordinates for the top-left corner of the text
cv.putText(final, left_text, (50, 50), font, font_scale, color, thickness, line_type)
cv.putText(final, right_text, (50, 90), font, font_scale, color, thickness, line_type)

show_img(final, title='FINAL WITH CLASSIFICATION')


In [ ]:
def fill_lanes(img, lines, color=(0, 255, 0), alpha=0.4):
    """
    Fill the lane region between two detected lane lines

    Args:
        img (np.ndarray): Original image
        lines (array-like): Two lane lines [left, right] in [x1, y1, x2, y2] format
        color (tuple): BGR color used to fill the lane region
        alpha (float): Transparency factor

    Returns:
        np.ndarray: Image with the lane region filled
    """
    # If lane lines are missing or there is just 1, return original image
    if lines is None or len(lines) < 2:
        print("Lane line Missing")
        return img

    # Create a blank image for drawing the filled lane
    overlay = np.zeros_like(img)

    # Ensure left/right ordering is correct
    l1, l2 = lines
    if l1[0] > l2[0]:
        l1, l2 = l2, l1

    # Points must be ordered correctly:
    # Bottom-left - top-left - top-right - bottom-right
    pts = np.array([
        [l1[0], l1[1]],  # left bottom
        [l1[2], l1[3]],  # left top
        [l2[2], l2[3]],  # right top
        [l2[0], l2[1]]   # right bottom
    ], dtype=np.int32)

    # Fill the polygon representing the lane area
    cv.fillPoly(overlay, [pts], color)

    # Blend the overlay with the original image
    return cv.addWeighted(img, 1, overlay, alpha, 0)

# --------------------------------------------------------
# Right Lane Area
# --------------------------------------------------------

avg = avg_lines(lane_img, lines)
lane_fill = fill_lanes(lane_img, avg)

show_img(lane_fill, title="Lane region")

In [ ]:
# --------------------------------------------------------
# FRAMES TAKEN FROM VIDEO 1
# --------------------------------------------------------

def get_frame(path, idx):
    """
    Extract a specific frame from a video file

    Args:
        path (str): Path to the video file
        idx (int): Index of the frame to extract

    Returns:
        np.ndarray: Extracted video frame
    """
    # Open the video file
    cap = cv.VideoCapture(path)

    # Check if the video was opened successfully
    if not cap.isOpened():
        print("Error")
        return

    # Get total number of frames in the video
    n_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

    # Validate requested frame index
    if idx >= n_frames:
        print(f"Error: Video has only {n_frames} frames.")
        return

    # Jump directly to the requested frame
    cap.set(cv.CAP_PROP_POS_FRAMES, idx)

    # Read the frame (first variable useless, could be _)
    temp, frame = cap.read()

    cap.release()
    return frame


# Extract a single frame from the video
frame = get_frame('roads/video1.mp4', 100)

# Display the extracted frame
show_img(frame, title="Frame 100")

# Create a frame copy for drawing lane lines
lane_img = np.copy(frame)

# --------------------------------------------------------
# Hough Line Detection as before
# --------------------------------------------------------

# We lower upper and lower bound to increase detectability of lines, otherwise it gets none
edges = canny_blur(frame, 50, 150)

h = lane_img.shape[0]

poly = np.array([
        [(200, h - 100), (1700, h - 100), (950, 530)]
        ])
edges_roi = roi(edges, poly)
# show_img(edges_roi) #debug

rho = 10
theta = np.pi / 180
thresh = 400

lines = cv.HoughLinesP(
    edges_roi,
    rho=rho,
    theta=theta,
    threshold=thresh,
    minLineLength=50,
    maxLineGap=20
)
line_img = draw_lines(lane_img, lines, 10)
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
show_img(all_lines_img, title='Lane lines')

avg = avg_lines(lane_img, lines)
line_img = draw_lines(frame, avg, 15)
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL')


In [ ]:
#LANE AREA as before

avg = avg_lines(lane_img, lines)

lane_fill = fill_lanes(lane_img, avg)
show_img(lane_fill, title="Lane Area")

In [ ]:
# ============================================================
# Line type classification for frame 100 of video 1
# ============================================================

# Separate detected Hough lines into left and right lanes
left_lines, right_lines = split_lines_by_side(lines)

# Classify left lane
left_type, left_ratio, left_gaps = classify_line_type(
    left_lines, frame
)

# Classify right lane
right_type, right_ratio, right_gaps = classify_line_type(
    right_lines, frame
)

print(f"Left lane:  {left_type} | hit_ratio={left_ratio:.2f}, gaps={left_gaps}")
print(f"Right lane: {right_type} | hit_ratio={right_ratio:.2f}, gaps={right_gaps}")


In [ ]:
# FRAME 400 of video 1, same as before
frame = get_frame('roads/video1.mp4', 400)
show_img(frame, title="Frame 400")

lane_img = np.copy(frame)

edges = canny_blur(frame, 50, 150)
edges_roi = roi(edges, poly)
# show_img(edges_roi) #debug

lines = cv.HoughLinesP(
    edges_roi,
    rho=rho,
    theta=theta,
    threshold=thresh,
    minLineLength=50,
    maxLineGap=20
)

line_img = draw_lines(lane_img, lines, 10)
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
show_img(all_lines_img, title='Lane lines')

avg = avg_lines(lane_img, lines)
line_img = draw_lines(frame, avg, 15)
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL')


In [ ]:
#ROAD AREA

avg = avg_lines(lane_img, lines)

lane_fill = fill_lanes(lane_img, avg)
show_img(lane_fill, title="Lane Area")

In [ ]:
# ============================================================
# Line type classification for frame 400 of video 1
# ============================================================

# Separate detected Hough lines into left and right lanes
left_lines, right_lines = split_lines_by_side(lines)

# Classify left lane
left_type, left_ratio, left_gaps = classify_line_type(
    left_lines, frame
)

# Classify right lane
right_type, right_ratio, right_gaps = classify_line_type(
    right_lines, frame
)

print(f"Left lane:  {left_type} | hit_ratio={left_ratio:.2f}, gaps={left_gaps}")
print(f"Right lane: {right_type} | hit_ratio={right_ratio:.2f}, gaps={right_gaps}")

In [ ]:
# --------------------------------------------------------
# FRAMES TAKEN FROM VIDEO 2
# --------------------------------------------------------

frame = get_frame('roads/video2.mp4', 200)
show_img(frame, title="Frame 200")

lane_img = np.copy(frame)

# re-define region of interest, only changes triangle
def roi_vid2(img):
    h = img.shape[0]

    poly = np.array([
        [(100, h - 150), (1900, h - 150), (850, 650)]
    ])

    mask = np.zeros_like(img)
    mask = cv.fillPoly(mask, poly, 255)
    mask = cv.bitwise_and(img, mask)

    return mask


# --------------------------------------------------------
# Hough Line Detection for entire roadway
# --------------------------------------------------------

edges = canny_blur(frame, 20, 100)
edges_roi = roi_vid2(edges)
show_img(edges_roi, title="Edges")

rho = 10
theta = np.pi / 180
thresh = 400

lines = cv.HoughLinesP(
    edges_roi,
    rho=rho,
    theta=theta,
    threshold=thresh,
    minLineLength=50,
    maxLineGap=20
)

line_img = draw_lines(lane_img, lines, 10)
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
show_img(all_lines_img, title='Lane lines')

avg = avg_lines(lane_img, lines)
line_img = draw_lines(frame, avg, 15)
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL')


In [ ]:
# Roadway Area
avg = avg_lines(lane_img, lines)

lane_fill = fill_lanes(lane_img, avg)
show_img(lane_fill, title="Roadway Area")

In [ ]:
# Same frame, computing just current Lane Region

# Extract a frame from the second video
frame = get_frame('roads/video2.mp4', 200)
#show_img(frame, title="frame200")

# Copy frame for drawing overlays
lane_img = np.copy(frame)

# --------------------------------------------------------
# Re-defined avg_lines to be more robust, only changes are commented                                                                                                                                                                                                                                      - only changes are commented
# --------------------------------------------------------
def avg_lines(img, lines):
    if lines is None:
        return None

    left = []
    right = []

    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)

        # AVOID VERTICAL LINES
        if x1 == x2:
            continue

        m, b = np.polyfit((x1, x2), (y1, y2), 1)

        if m < 0:
            left.append((m, b))
        else:
            right.append((m, b))

    # if one side is missing, do NOT average
    if len(left) == 0 or len(right) == 0:
        print("One side lane is missing")
        return None
        
    left_avg = np.mean(left, axis=0)
    right_avg = np.mean(right, axis=0)

    left_line = line_coords(img, left_avg)
    right_line = line_coords(img, right_avg)

    return np.array([left_line, right_line])

#Re-defined region of interest
def roi_vid2(img):
    h, w = img.shape[:2]

    poly = np.array([[
        (int(0.05 * w), (h - 145)),
        (int(0.7 * w), (h - 145)),
        (int(0.42 * w), int(0.63 * h))
    ]])

    mask = np.zeros_like(img)
    mask = cv.fillPoly(mask, poly, 255)
    mask = cv.bitwise_and(img, mask)

    return mask


# --------------------------------------------------------
# Hough Line Detection
# --------------------------------------------------------

edges = canny_blur(frame, 50, 150)

edges_roi = roi_vid2(edges)
show_img(edges_roi, title = "Edges")

lines = cv.HoughLinesP(
    edges_roi,
    rho=10,                     # finer resolution than before
    theta=np.pi / 180,
    threshold=120,              # decreased to get more lines
    minLineLength=50,           
    maxLineGap=50               
)

line_img = draw_lines(lane_img, lines, 10)
all_lines_img = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)
avg = avg_lines(lane_img, lines)

line_img = draw_lines(frame, avg, 15)
final = cv.addWeighted(lane_img, 0.8, line_img, 1, 1)

show_img(final, title='FINAL - Lane Lines')


In [ ]:
# --------------------------------------------------------
# Car Detection
# --------------------------------------------------------

def detect_cars_with_roi_constraint(img):
    """
    Detect cars using Harris Corner Detection restricted to a Right-Side ROI.
    The final output is the full image with boxes only in that region.
    """
    h, w = img.shape[:2]
    output_img = img.copy()

    # Define the ROI Coordinates (Right side, lower half)
    roi_x = int(w * 0.5)
    roi_y_start = int(h * 0.4)
    roi_y_end = int(h * 0.9)
    
    # Create the cropped ROI for processing
    roi = img[roi_y_start:roi_y_end, roi_x:w]
    
    # Harris Corner Detection on the ROI ONLY
    gray_roi = cv.cvtColor(roi, cv.COLOR_BGR2GRAY)
    gray_roi = np.float32(gray_roi) #returns an error oif we don't perform this
    
    # Detect corners
    dst = cv.cornerHarris(gray_roi, 2, 3, 0.04)
    dst = cv.dilate(dst, None)
    
    # Create a binary mask where corners are significant
    corner_mask = np.zeros_like(gray_roi, dtype=np.uint8)
    corner_mask[dst > 0.015 * dst.max()] = 255
    
    # Morphological grouping to merge corners into car-sized blobs
    kernel = np.ones((25, 25), np.uint8)
    closing = cv.morphologyEx(corner_mask, cv.MORPH_CLOSE, kernel)
    
    # Find contours in the processed ROI
    contours, _ = cv.findContours(closing, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    
    for cnt in contours:
        x, y, cw, ch = cv.boundingRect(cnt)
        area = cv.contourArea(cnt)
        aspect_ratio = float(cw) / ch
        
        # Filter for car-like dimensions within the ROI
        # Area and aspect ratio help distinguish cars from road marks
        if 1000 < area < 50000:
            if 0.7 < aspect_ratio < 2.5:
                # MAP COORDINATES BACK TO THE FULL IMAGE
                # We add the ROI start offsets to the local x and y
                full_x = x + roi_x
                full_y = y + roi_y_start
                
                # Draw the detection on the FULL image
                cv.rectangle(output_img, (full_x, full_y), (full_x + cw, full_y + ch), (0, 0, 255), 3)
                cv.putText(output_img, 'Car', (full_x, full_y - 10), 
                           cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    
    return output_img
#----------------
# We use FRAME 100 of video 1
#----------------

frame = get_frame('roads/video1.mp4', 100)

if frame is not None:
    result_frame = detect_cars_with_roi_constraint(frame)
    
    show_img(result_frame, title="Car Detection")
else:
    print("Error")

In [ ]:
#VIDEO LANE DETECTION
from IPython.display import display, clear_output # To display video efficiently

# -----------------------------
# Region of Interest for video (a little bit more restrictive than before, works better on the video)
# -----------------------------
def roi_vid2(img):
    h, w = img.shape[:2]
    poly = np.array([[
        (int(0.2 * w), h - 145),
        (int(0.8 * w), h - 145),
        (int(0.44 * w), int(0.61 * h))
    ]])
    mask = np.zeros_like(img)
    cv.fillPoly(mask, poly, 255)
    return cv.bitwise_and(img, mask)

# -----------------------------
# Draw Hough lines ON FRAME
# -----------------------------
def draw_lines(img, lines):
    if lines is not None:
        for line in lines.reshape(-1, 4):
            x1, y1, x2, y2 = line
            cv.line(img, (x1, y1), (x2, y2), (0, 255, 0), 4)
    return img

# -----------------------------
# REDEFINE TO LOWER HIT RATIO
# -----------------------------

def classify_line_type(lines, img, n_checkLines=50):
   
    yc = checkLine_yc(img, n=n_checkLines)
    hits = checkLine_intersects(lines, yc)
    hit_ratio = hits.mean()
    gap_count = np.sum(~hits)
    if hit_ratio > 0.5:
        return "solid", hit_ratio, gap_count
    else:
        return "dashed", hit_ratio, gap_count

#----------------------------------------

# Hough parameters
rho = 2
theta = np.pi / 180
thresh = 100
min_line_len = 30
max_line_gap = 10
frame_skip = 1 #for efficiency, otherwise the loop would be way too slow

cap = cv.VideoCapture("roads/video2.mp4")
frame_count = 0

# -----------------------------
# Video loop
# -----------------------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    #show one every two frames
    frame_count += 1
    if frame_count % frame_skip != 0:
        continue

    # Edge detection - changed a bit high_threshold, works well even with the previous 100
    edges = canny_blur(frame, 20, 150)
    cropped = roi_vid2(edges)

    # Hough transform
    lines = cv.HoughLinesP(
        cropped,
        rho=rho,
        theta=theta,
        threshold=thresh,
        minLineLength=min_line_len,
        maxLineGap=max_line_gap
    )

    output = frame.copy() #same as lane_img

    if lines is not None:
        
        # Lane classification
        left_lines, right_lines = split_lines_by_side(lines)

        left_type, left_ratio, left_gaps = classify_line_type(
            left_lines, frame
        )

        right_type, right_ratio, right_gaps = classify_line_type(
            right_lines, frame
        )

        # Print in video lane type + hit ratio
        cv.putText(
            output,
            f"Left lane: {left_type} | hit_ratio={left_ratio:.2f}",
            (40, 50),
            cv.FONT_HERSHEY_SIMPLEX,
            1.3,
            (255, 0, 0),
            2
        )

        cv.putText(
            output,
            f"Right lane: {right_type} | hit_ratio={right_ratio:.2f}",
            (40, 100),
            cv.FONT_HERSHEY_SIMPLEX,
            1.3,
            (0, 0, 255),
            2
        )

        # Draw detected Hough lines on the frame
        output = draw_lines(output, lines)

    # -----------------------------
    # Display frame with a different method for efficiency
    # -----------------------------
    clear_output(wait=True)
    plt.imshow(cv.cvtColor(output, cv.COLOR_BGR2RGB))
    plt.axis("off")
    display(plt.gcf())

cap.release()
plt.close()